# Frozen benchmark set

Issue: #16

## Question

The LLM judge (#39) and the methods comparison need a small, fixed set of queries that never changes between model versions. Which ~300 pairs should it hold?

## Data used

Input: `data/eval/ranking_queries.parquet` (#10, 75,996 queries from the w0 evaluation days).

Output: `data/eval/benchmark_v1.parquet` and `data/eval/benchmark_v1.json` (version, seed, size, SHA-256 content hash, composition). Code: `notebooks/benchmark.py`.

The pairs carry the same known leak as the ranking queries: the query category comes from the held-out click.

## Allocation rule (written before looking at results)

- About 300 pairs. Each of the 80 categories gets an equal share (3 or 4 pairs); the extra 60 pairs go to categories picked by a seeded shuffle.
- Within those picks, the four prefix-length buckets (1, 2-3, 4-10, 11+) are kept as equal as possible, about 75 each. The data itself is 44% one-event prefixes, so this deliberately over-represents long sessions.
- **Fallback:** if a category has no query left in a bucket, the least-used other bucket with queries is used.
- One pair per session. Fixed seed 0.

In [ ]:
import json
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from benchmark import BUCKETS, build_benchmark, content_hash
from split import WINDOWS, day_to_ms

DATA = Path("data") if Path("data").exists() else Path("../data")
OUT = DATA / "eval"
VERSION = "v1"
SEED = 0
W = WINDOWS["w0 initial (weeks 1-2)"]
EVAL_START, EVAL_END = day_to_ms(W["eval_start_day"]), day_to_ms(W["eval_end_day"])

queries = pl.read_parquet(OUT / "ranking_queries.parquet")
print("queries available:", queries.height)

## Build it, and check that a re-run gives the same hash

In [ ]:
bench = build_benchmark(queries, n=300, seed=SEED)
h1 = content_hash(bench)
h2 = content_hash(build_benchmark(queries, n=300, seed=SEED))
assert h1 == h2
h_other = content_hash(build_benchmark(queries, n=300, seed=SEED + 1))
assert h_other != h1, "a different seed should give a different set"
print(f"{bench.height} pairs, content hash {h1}")
print("re-run gives the same hash; a different seed does not")

## Checks

In [ ]:
assert 280 <= bench.height <= 320
assert bench["session"].n_unique() == bench.height

# every pair comes from the evaluation window: the session starts inside the eval days, and the prefix ends before eval_end
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
firsts = (
    events.filter(pl.col("session").is_in(bench["session"].to_list()))
    .group_by("session").agg(pl.col("ts").min().alias("first_ts")).collect()
)
assert firsts.height == bench.height
assert (firsts["first_ts"] >= EVAL_START).all() and (firsts["first_ts"] < EVAL_END).all()
assert (bench["prefix_ts_last"] < EVAL_END).all()
# so no benchmark session has any event in the training window (no overlap with ranker training sessions)
print("every pair comes from the evaluation window; no event of any benchmark session is before the eval start")

n_cat = queries["query_category"].n_unique()
per_cat = bench.group_by("query_category").len()
print(f"categories covered: {per_cat.height} of {n_cat}")
assert per_cat.height == n_cat, "some category is missing"

## Composition

In [ ]:
per_cat = bench.group_by("query_category").len().sort("query_category")
print("pairs per category:", per_cat["len"].value_counts().sort("len").rows())

by_bucket = bench.group_by("prefix_bucket").len()
by_bucket = by_bucket.with_columns(pl.col("prefix_bucket").replace_strict({b: i for i, b in enumerate(BUCKETS)}).alias("o")).sort("o").drop("o")
data_share = (
    queries.select(pl.col("prefix_aid").list.len().alias("n"))
    .with_columns(pl.col("n").map_elements(lambda n: "1" if n <= 1 else "2-3" if n <= 3 else "4-10" if n <= 10 else "11+", return_dtype=pl.Utf8).alias("prefix_bucket"))
    .group_by("prefix_bucket").len().with_columns((pl.col("len") / queries.height).alias("share_in_data")).select("prefix_bucket", "share_in_data")
)
print(by_bucket.join(data_share, on="prefix_bucket").with_columns((pl.col("len") / bench.height).alias("share_in_benchmark")))

pair_table = bench.group_by("query_category", "prefix_bucket").len().pivot(on="prefix_bucket", index="query_category", values="len").fill_null(0).sort("query_category")
print(pair_table.head(10))

In [ ]:
bench.write_parquet(OUT / f"benchmark_{VERSION}.parquet")
meta = {
    "version": VERSION, "seed": SEED, "n": bench.height, "sha256": h1,
    "source": "ranking_queries.parquet (window w0 evaluation days)",
    "allocation": "equal per category; prefix-length buckets as equal as possible; one pair per session",
    "pairs_per_bucket": dict(bench.group_by("prefix_bucket").len().iter_rows()),
    "categories": int(per_cat.height),
    "known_leak": "query category comes from the held-out click (see issue #10)",
}
(OUT / f"benchmark_{VERSION}.json").write_text(json.dumps(meta, indent=2))
again = pl.read_parquet(OUT / f"benchmark_{VERSION}.parquet")
assert content_hash(again) == h1
print(f"version {VERSION}, {bench.height} pairs, sha256 {h1}")

## Result

The set is frozen as version v1 with the hash printed above. Never edit it: make `benchmark_v2` if it must change, so older results stay comparable.

## What I learned

To be written by the owner of the project.